# P05 — Gaussian basis และ regularization
ส่วนเสริมจาก Copy_of_Lecture_09_Regression_Student_Exercise.ipynb
Notebook นี้แยกจากต้นฉบับ ใช้ข้อมูลจำลองเท่านั้น และกำหนด protocol ชัดเจนเพื่อเทียบผลอย่างยุติธรรม

เป้าหมาย: สร้างข้อมูลจากสมการ แปลง Polynomial/Gaussian features แล้วเปรียบเทียบ OLS/Ridge/Lasso บน basis และสเกลเดียวกัน
ติดตั้งแพ็กเกจตาม requirements.txt ก่อนรัน ข้อมูลสุ่มสร้างในเครื่อง ไม่ต้องดาวน์โหลดข้อมูล


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error

def rmse(y, prediction):
    return np.sqrt(mean_squared_error(y, prediction))


## เครื่องมือสร้างข้อมูลและ Gaussian basis
ฟังก์ชันด้านล่างใช้ σ = width_factor × ระยะห่าง centers; fit ต้องรับเฉพาะ train
กรณี N<2, width_factor≤0 หรือ train มี x ค่าเดียวจะให้ข้อความผิดพลาด แทนการหารด้วยศูนย์


In [ ]:
"""Portable, one-dimensional Gaussian feature protocol for P05 (synthetic data)."""
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted

def make_sine_data(seed, n):
    """Independent RNG per split; all randomness comes from this generator."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 10, size=(n, 1))
    y = np.sin(x[:, 0]) + rng.normal(0, 0.15, size=n)
    return x, y

class GaussianFeatures(TransformerMixin, BaseEstimator):
    """N train-fitted centers; sigma=width_factor*center spacing (not factor alone)."""
    def __init__(self, N=12, width_factor=0.8):
        self.N = N
        self.width_factor = width_factor

    def fit(self, X, y=None):
        X = check_array(X, ensure_2d=True)
        if X.shape[1] != 1:
            raise ValueError('P05 protocol accepts one input feature only.')
        if not isinstance(self.N, (int, np.integer)) or self.N < 2:
            raise ValueError('N must be an integer >= 2.')
        if not np.isfinite(self.width_factor) or self.width_factor <= 0:
            raise ValueError('width_factor must be positive and finite.')
        if X.max() <= X.min():
            raise ValueError('Training x must have a positive range.')
        self.n_features_in_ = 1
        self.centers_ = np.linspace(X.min(), X.max(), self.N)
        self.sigma_ = self.width_factor * (self.centers_[1] - self.centers_[0])
        return self

    def transform(self, X):
        check_is_fitted(self, ['centers_', 'sigma_'])
        X = check_array(X, ensure_2d=True)
        if X.shape[1] != 1:
            raise ValueError('P05 protocol accepts one input feature only.')
        return np.exp(-0.5 * ((X[:, 0, None] - self.centers_[None, :]) / self.sigma_) ** 2)


## 1. สร้างข้อมูลตามสมการและแสดง Polynomial basis
- y=sin(x)+ε, x~Uniform(0,10), ε มีการแจกแจงปกติ ค่าเฉลี่ย0 ส่วนเบี่ยงเบนมาตรฐาน0.15
- train50 seed41, validation100 seed42, finaltest100 seed43 ใช้ generator แยกตาม seed
- แสดง PolynomialFeatures(degree=3,include_bias=False) ของ train 3 แถวแรก
ยังไม่ประเมิน final test จนกว่าจะเลือกด้วย validation


In [ ]:
# TODO: ใช้ make_sine_data สร้าง train/validation ตาม protocol
# TODO: สร้างและแสดง polynomial feature matrix ของ 3 แถวแรก


## 2. Fit basis และสเกลครั้งเดียวด้วย train
- GaussianFeatures(N=12,width_factor=0.8)
- รายงาน centers_, sigma_ และขนาด feature matrix; วาดกราฟ basis
- fit StandardScaler ด้วย Gaussian features ของ train แล้วใช้ scaler ตัวเดิม transform validation
ทั้งสามโมเดลต้องได้รับ feature matrix ที่เหมือนกัน


In [ ]:
# TODO: fit_transform basis/scaler ด้วย train เท่านั้น
# TODO: transform validation; รายงานค่าและวาดกราฟ


## 3. เปรียบเทียบและเลือกโมเดล
- OLS = LinearRegression()
- Ridge(alpha=0.1)
- Lasso(alpha=0.001,max_iter=100000,tol=1e-6)
รายงาน train/validation RMSE; เลือก validation RMSE ต่ำสุด หากต่างไม่เกิน1e-6 ใช้ลำดับ OLS,Ridge,Lasso เลือกตัวแรก
จึงสร้าง/ประเมิน final test100 ด้วย seed43 สำหรับตัวที่เลือกเพียงแบบเดียว โดยใช้โมเดล/basis/scalerที่fitบนtrainเดิม ไม่ refit และไม่ย้อนเลือกจาก test


In [ ]:
# TODO: fit สามโมเดลบน Gaussian scaled train matrix เดียวกัน
# TODO: ตารางผล train/validation, เลือกแบบ และ final-test RMSE ของตัวเลือกเดียว


## 4. อธิบายผลด้วยหลักฐาน
- penalty เปลี่ยนผลในตารางอย่างไร ไม่ต้องบังคับให้ Ridge/Lasso ชนะ
- เหตุใดค่า alpha ตัวเลขเดียวกันข้าม Ridge/Lasso จึงไม่ได้แปลว่าแรง penalty เท่ากัน
- ถ้าเปลี่ยน N พร้อมชนิด penalty จะมีตัวแปรกวนอย่างไร
- seed ช่วยทำซ้ำได้ แต่ไม่ทำให้ noise หายหรือ sample correlation สมบูรณ์แบบ
